# Search Tools

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


import wikipedia
from ddgs import DDGS
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Define tools (DuckDuckGo + Wikipedia)

In [ ]:
wikipedia.set_lang("en")


def _format_hits(hits: list[dict], k: int = 5) -> str:
    lines = []
    for i, h in enumerate(hits[:k], 1):
        title = h.get("title") or h.get("heading") or ""
        url = h.get("href") or h.get("url") or ""
        snippet = h.get("body") or h.get("description") or h.get("snippet") or ""
        lines.append(f"{i}. {title}\n   {url}\n   {snippet}")
    return "\n".join(lines) if lines else "No results."


@tool
def web_search(query: str, max_results: int = 5, time_range: str | None = None) -> str:
    """DuckDuckGo web search. Returns up to `max_results` compact results with URLs and snippets.
    time_range can be 'd','w','m','y' (day, week, month, year) for recency, or None."""
    try:
        with DDGS() as ddg:
            hits = list(ddg.text(query, max_results=max_results, timelimit=time_range))
        return _format_hits(hits, k=max_results)
    except Exception as e:
        return f"Search error: {e}"


@tool
def wiki_summary(topic: str, sentences: int = 3) -> str:
    """Return a short Wikipedia summary with the canonical page URL."""
    try:
        page = wikipedia.page(topic, auto_suggest=True, redirect=True)
        summary = wikipedia.summary(page.title, sentences=sentences)
        return f"Title: {page.title}\nURL: {page.url}\nSummary: {summary}"
    except wikipedia.DisambiguationError as e:
        opts = "; ".join(e.options[:5])
        return f"Disambiguation: be specific. Options include: {opts}"
    except wikipedia.PageError:
        return "No matching page found."
    except Exception as e:
        return f"Wikipedia error: {e}"

# Build the search agent

In [ ]:
def build_search_agent():
    llm = ChatOpenAI(
        model=os.getenv("GPT_MODEL", "your-gpt-model"),
        base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
        api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
    )
    tools = [web_search, wiki_summary]

    SYSTEM_PROMPT = (
        "You are a research assistant. Today's date is {today}.\n"
        "RULES:\n"
        "1) Use tools for ANY factual claim, number, date, or 'latest' info. Never answer these from memory.\n"
        "2) Tool choice:\n"
        "   - web_search: news, recent events, or when URLs are required. For 'recent', 'this week' or 'latest', "
        "set time_range ('d' = day, 'w' = week, 'm' = month).\n"
        "   - wiki_summary: background and definitions.\n"
        "3) Cite only titles and URLs that appear in tool outputs. Never invent or edit a URL.\n"
        "4) For news requests, return individual articles, not section or topic index pages. "
        "If the results are index pages, search again with a more specific query.\n"
        "5) If the tools return nothing useful, say so instead of guessing.\n"
        "6) Be concise. Use at most the top 3 results, and follow any output format the user asks for."
    )

    return create_agent(llm, tools=tools, system_prompt=SYSTEM_PROMPT)


search_agent = build_search_agent()


def ask(agent, query: str) -> str:
    return agent.invoke({"messages": [("human", query)]})["messages"][-1].content


print(" Search agent ready.")

 Search agent ready.


# Quick tests

In [4]:
# 1) Simple fact + citation
print(
    ask(
        search_agent,
        "Who won the FIFA World Cup 2022? Give one sentence and include a source URL.",
    )
)

# 2) Background from Wikipedia
print(
    ask(
        search_agent,
        "In 3 bullets, explain Retrieval-Augmented Generation. Cite the Wikipedia page with URL.",
    )
)

# 3) Fresh headlines (limit and summarize)
print(
    ask(
        search_agent,
        "Find 3 recent headlines about generative AI this week. One line per item with URL.",
    )
)

Argentina won the 2022 FIFA World Cup, defeating France on penalties after a 3-3 draw. 

Source: Who won the last FIFA World Cup? Complete story of the 2022 final and ... https://economictimes.indiatimes.com/news/new-updates/who-won-the-last-fifa-world-cup-2022-winner-final-score-know-about-everything-argentina-vs-france-kylian-mbappe-lionel-messi/articleshow/132490160.cms
- Retrieval-Augmented Generation (RAG) combines a retrieval module with a generator, enabling the model to fetch relevant documents from an external corpus during inference.
- Retrieved documents supplement the model’s training data, helping it produce more up-to-date or domain-specific responses.
- The typical workflow is: retrieve relevant passages, encode them with the query, and condition the generation on the retrieved content.

Source: Retrieval-augmented generation — https://en.wikipedia.org/wiki/Retrieval-augmented_generation
- This week in the big AI data center buildout — https://www.theverge.com/ai-artific